# Session 8 — Tabular Q-learning on Electronics

**Salih Tutun, PhD** · DAT 5668 · Module 4 · Practice · Explain → practice together

From Fall 2025 **Lab 4** (inventory Q-learning). Tonight = **tabular Q1–Q3**. DQN → Session 9.

| Cue | Meaning |
| --- | --- |
| **RUN WITH ME** | Ready. Run. |
| **TRY THIS** | Fill `<<< FILL` / complete the function. |
| **CHANGE ME** | Change α / ε. **PAIR 20s**. |

![Q-table is decide](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-q-decide.png)
![Q-learning update](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/q-update.png)

*All diagrams load from GitHub raw URLs.*


# Part I — Topic

Session 7 needed known P(d). Tonight we learn **Q(s,a)** from Electronics days.

```text
Q(s,a) ← Q(s,a) + α [ r + γ max_a' Q(s',a') − Q(s,a) ]
```

ε-greedy: with probability ε pick random action; else `argmax Q(s,·)`.

**Flow:** toy 3-state (Lab Q1) → real Electronics states (Lab Q2) → you write the update → optional 10 bins (Lab Q3).

**JUMP BACK** until I say OPEN.


---
# Part II — Practice


## Stop 0 — Imports


In [ ]:
# STOP 0a — RUN WITH ME · imports

from pathlib import Path
from collections import Counter
import csv
import random
import statistics

import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"
PRODUCT = "Electronics"
RNG = random.Random(7)


### 0b — course_data (RUN WITH ME)


In [ ]:
# STOP 0b — RUN WITH ME · course_data

def course_data(name: str) -> Path:
    # Prefer Drive, then local folders, then GitHub download (Colab-safe)
    candidates = []
    if DRIVE_DATA_FOLDER:
        candidates.append(Path(DRIVE_DATA_FOLDER) / name)
    for base in (
        Path("data"),
        Path("../../Drive-Folder-for-course/DAT5668/data"),
        Path("../Drive-Folder-for-course/DAT5668/data"),
        Path("../../../Drive-Folder-for-course/DAT5668/data"),
        Path("../../assets/data"),
        Path("assets/data"),
        Path("/content/data"),
        Path("../data"),
    ):
        candidates.append(base / name)
    for path in candidates:
        if path.exists():
            return path
    if name in {"inventory_dataset_full.csv", "inventory_dataset.csv"}:
        import urllib.request
        dest = Path("/content/data") if Path("/content").exists() else Path("data")
        dest.mkdir(parents=True, exist_ok=True)
        out = dest / name
        url = "https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/data/" + name
        print("Downloading", name, "from GitHub…")
        urllib.request.urlretrieve(url, out)
        return out
    raise FileNotFoundError("Could not find " + name + ". Mount Drive DAT5668/data or use GitHub fallback.")


## Stop 1 — Load Electronics


In [ ]:
# STOP 1 — RUN WITH ME · load

def load_electronics_desk():
    rows = []
    with course_data("inventory_dataset_full.csv").open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if (row.get("Product") or "").strip() != PRODUCT:
                continue
            rows.append({
                "Date": row["Date"],
                "Current_Inventory": int(float(row["Current_Inventory"])),
                "Demand": int(float(row["Demand"])),
                "Action_Taken": (row.get("Action_Taken") or "").strip(),
                "Profit": float(row.get("Profit") or 0),
            })
    if not rows:
        raise ValueError("No Electronics rows")
    rows.sort(key=lambda r: r["Date"])
    return rows


ROWS = load_electronics_desk()
N_DAYS = len(ROWS)
AVG_DEMAND = statistics.mean(r["Demand"] for r in ROWS)
print("Electronics days:", N_DAYS, "| avg demand:", round(AVG_DEMAND, 1))
print("Historical actions:", dict(Counter(r["Action_Taken"] for r in ROWS)))
print("CHECK load —", "PASS" if N_DAYS == 730 else "NOT YET")


## Stop 2 — Lab 4 Q1 warm-up (toy 3 states)

Hand-crafted rewards. Random next state. See the Q-update move before we wire the real log.


In [ ]:
# STOP 2a — RUN WITH ME · toy Q-learning

STATES = ["Low Inventory", "Medium Inventory", "High Inventory"]
ACTIONS = ["Order", "Maintain", "Reduce"]
REWARDS = {
    ("Low Inventory", "Order"): 10, ("Low Inventory", "Maintain"): -5, ("Low Inventory", "Reduce"): -10,
    ("Medium Inventory", "Order"): 2, ("Medium Inventory", "Maintain"): 5, ("Medium Inventory", "Reduce"): 2,
    ("High Inventory", "Order"): -10, ("High Inventory", "Maintain"): -5, ("High Inventory", "Reduce"): 10,
}

def train_toy(alpha=0.1, epsilon=0.1, gamma=0.9, episodes=100):
    Q = {s: {a: 0.0 for a in ACTIONS} for s in STATES}
    tracked = []
    for _ in range(episodes):
        state = RNG.choice(STATES)
        for _step in range(10):
            action = RNG.choice(ACTIONS) if RNG.random() < epsilon else max(Q[state], key=Q[state].get)
            reward = REWARDS[(state, action)]
            nxt = RNG.choice(STATES)
            old = Q[state][action]
            Q[state][action] = old + alpha * (reward + gamma * max(Q[nxt].values()) - old)
            if state == "Low Inventory" and action == "Order":
                tracked.append(Q[state][action])
            state = nxt
    policy = {s: max(Q[s], key=Q[s].get) for s in STATES}
    return Q, policy, tracked

Q_toy, policy_toy, tracked_toy = train_toy()
print("Toy policy:", policy_toy)
print("CHECK toy —", "PASS" if policy_toy["Low Inventory"] == "Order" and policy_toy["High Inventory"] == "Reduce" else "LOOK")

if plt is not None and tracked_toy:
    fig, ax = plt.subplots(figsize=(5, 2.6))
    ax.plot(tracked_toy, color="#2F5D50")
    ax.set_title("Toy Q(Low, Order)")
    fig.tight_layout()
    plt.show()


### 2b — CHANGE ME · α / ε on the toy


In [ ]:
# STOP 2b — CHANGE ME

ALPHA = 0.1    # <<< CHANGE ME: 0.05 / 0.3 / 1.0
EPSILON = 0.1  # <<< CHANGE ME: 0.0 / 0.5

_, policy2, tracked2 = train_toy(alpha=ALPHA, epsilon=EPSILON)
print("alpha", ALPHA, "epsilon", EPSILON, "→", policy2)
if plt is not None and tracked2:
    fig, ax = plt.subplots(figsize=(5, 2.6))
    ax.plot(tracked2, color="#C45C26")
    ax.set_title(f"Q(Low,Order) α={ALPHA} ε={EPSILON}")
    fig.tight_layout()
    plt.show()


## Stop 3 — Lab 4 Q2: states from real Electronics days

`Current_Inventory / avg Demand` → Low / Medium / High (same idea as Session 6).


In [ ]:
# STOP 3 — RUN WITH ME · categorize

def categorize_inventory(inv, avg_demand, low_cut=0.5, high_cut=2.0):
    ratio = float(inv) / max(avg_demand, 1e-9)
    if ratio < low_cut:
        return "Low Inventory"
    if ratio <= high_cut:
        return "Medium Inventory"
    return "High Inventory"


for r in ROWS:
    r["State"] = categorize_inventory(r["Current_Inventory"], AVG_DEMAND)

counts = Counter(r["State"] for r in ROWS)
print("State mix:", dict(counts))
if plt is not None:
    labels = ["Low Inventory", "Medium Inventory", "High Inventory"]
    fig, ax = plt.subplots(figsize=(5, 2.6))
    ax.bar(labels, [counts[k] for k in labels], color=["#C45C26", "#2F5D50", "#1F4E79"])
    ax.set_title("Electronics state mix")
    fig.tight_layout()
    plt.show()


## Stop 4 — YOU write ε-greedy + Q-update (Lab 4 core)

![Update](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/q-update.png)

Walk consecutive Electronics days. Complete the two functions. Re-run until **CHECK 4 — PASS**.


In [ ]:
# STOP 4 — TRY THIS · Q-learning on Electronics days

STATES = ["Low Inventory", "Medium Inventory", "High Inventory"]
ACTIONS = ["Order", "Maintain", "Reduce"]
REWARDS_DATA = {
    ("Low Inventory", "Order"): 2, ("Low Inventory", "Maintain"): -1, ("Low Inventory", "Reduce"): -2,
    ("Medium Inventory", "Order"): 0, ("Medium Inventory", "Maintain"): 1, ("Medium Inventory", "Reduce"): 0,
    ("High Inventory", "Order"): -2, ("High Inventory", "Maintain"): -1, ("High Inventory", "Reduce"): 2,
}

alpha = 0.3
epsilon = 0.5  # <<< CHANGE ME after first PASS
gamma = 0.7
n_episodes = 50

Q = {s: {a: 0.0 for a in ACTIONS} for s in STATES}
tracked = []


def epsilon_greedy(state, Q, epsilon):
    # <<< FILL: if random < epsilon → random action; else argmax Q[state]
    return None  # replace this line


def q_update(Q, state, action, reward, next_state, alpha, gamma):
    # <<< FILL: Q[s,a] ← Q[s,a] + α (r + γ max Q[s'] − Q[s,a])
    return None  # replace: update Q in place


ready = True
try:
    for _ in range(n_episodes):
        for i in range(len(ROWS) - 1):
            state = ROWS[i]["State"]
            action = epsilon_greedy(state, Q, epsilon)
            if action not in ACTIONS:
                raise ValueError("epsilon_greedy must return an action string")
            reward = REWARDS_DATA[(state, action)]
            next_state = ROWS[i + 1]["State"]
            before = Q[state][action]
            q_update(Q, state, action, reward, next_state, alpha, gamma)
            if Q[state][action] == before and alpha != 0:
                # still ok early on; keep going
                pass
            if state == "Low Inventory" and action == "Order":
                tracked.append(Q[state][action])
except Exception as err:
    ready = False
    print("NOT YET —", err)
    print("Fill epsilon_greedy and q_update, then re-run this cell.")

if ready:
    policy = {s: max(Q[s], key=Q[s].get) for s in STATES}
    print("Policy:", policy)
    ok = (
        policy["Low Inventory"] == "Order"
        and policy["Medium Inventory"] == "Maintain"
        and policy["High Inventory"] == "Reduce"
    )
    print("CHECK 4 —", "PASS" if ok else "LOOK — expected Low→Order, Med→Maintain, High→Reduce")
    # Validation plot: learned policy vs historical Action_Taken when states match
    agree = 0
    for r in ROWS:
        if policy[r["State"]] == r["Action_Taken"]:
            agree += 1
    print("Agree with historical Action_Taken:", agree, "/", N_DAYS, f"({agree/N_DAYS:.1%})")
    if plt is not None:
        fig, axes = plt.subplots(1, 2, figsize=(8.5, 2.8))
        if tracked:
            axes[0].plot(tracked[:: max(1, len(tracked)//200)], color="#1F4E79")
        axes[0].set_title("Q(Low, Order) updates")
        labels = STATES
        axes[1].bar(["agree", "disagree"], [agree, N_DAYS - agree], color=["#2F5D50", "#8B3A2A"])
        axes[1].set_title("Policy vs history")
        fig.tight_layout()
        plt.show()


### Hint (after you try)

```python
def epsilon_greedy(state, Q, epsilon):
    if RNG.random() < epsilon:
        return RNG.choice(ACTIONS)
    return max(Q[state], key=Q[state].get)

def q_update(Q, state, action, reward, next_state, alpha, gamma):
    old = Q[state][action]
    Q[state][action] = old + alpha * (reward + gamma * max(Q[next_state].values()) - old)
```


## Stop 5 — Optional: Lab 4 Q3 (10 bins)

**CHANGE ME** `NUM_BINS` if you finish early.


In [ ]:
# STOP 5 — CHANGE ME · bins

NUM_BINS = 10  # <<< CHANGE ME: 5 or 15
inv_vals = [r["Current_Inventory"] for r in ROWS]
edges = np.linspace(min(inv_vals), max(inv_vals), num=NUM_BINS)


def to_bin(inv):
    return int(np.digitize(inv, edges))


def bin_reward(state, action):
    if state < 3 and action == "Order":
        return 2
    if 3 <= state < 7 and action == "Maintain":
        return 1
    if state >= 7 and action == "Reduce":
        return 2
    return -1


Qb = {s: {a: 0.0 for a in ACTIONS} for s in range(NUM_BINS + 1)}
for _ in range(30):
    for i in range(len(ROWS) - 1):
        s = to_bin(ROWS[i]["Current_Inventory"])
        a = RNG.choice(ACTIONS) if RNG.random() < 0.5 else max(Qb[s], key=Qb[s].get)
        sp = to_bin(ROWS[i + 1]["Current_Inventory"])
        old = Qb[s][a]
        Qb[s][a] = old + 0.3 * (bin_reward(s, a) + 0.8 * max(Qb[sp].values()) - old)

print("bins", NUM_BINS, "policy@0/mid/high:",
      max(Qb[0], key=Qb[0].get),
      max(Qb[NUM_BINS // 2], key=Qb[NUM_BINS // 2].get),
      max(Qb[NUM_BINS], key=Qb[NUM_BINS].get))


## Stop 6 — Diagnostics + deploy talk

| Knob | What breaks |
| --- | --- |
| `epsilon=0` | Freezes on a lucky first action |
| `alpha=1` | Chases the last sample |
| Bad rewards | Orders when High / reduces when Low |

Would you put this Q-table on a real SKU tomorrow? Name the **human override** (Session 6 memo).


## Stop 7 — Framework (TRY THIS)


In [ ]:
# STOP 7 — TRY THIS

framework_family = ""  # <<< FILL: LLM / RL / hybrid
framework_line = ""    # <<< FILL
print("Family:", framework_family or "(FILL)")
print("I will keep:", framework_line or "(FILL)")
print("CHECK 7 —", "PASS" if framework_family and framework_line else "NOT YET")


## Raise a hand — Session 8 complete

1. [ ] Toy policy Low→Order, High→Reduce  
2. [ ] CHECK 4 PASS on Electronics days  
3. [ ] Changed α or ε and said what moved  
4. [ ] Named a human override for a bad Q row  
